# Advanced Mutual Fund Analytics

## 1. Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load datasets
nav_df = pd.read_csv('data/processed/nav_history_clean.csv', parse_dates=['date'])
fm_df = pd.read_csv('data/processed/fund_master_clean.csv')
perf_df = pd.read_csv('data/processed/scheme_performance_clean.csv')
txn_df = pd.read_csv('data/processed/investor_transactions_clean.csv', parse_dates=['transaction_date'])
hld_df = pd.read_csv('data/processed/portfolio_holdings_clean.csv')


## 2. Data Validation

In [ ]:
# Validated during Phase 0: Data Profiling
print("Data validation completed successfully.")
print(f"Total funds: {fm_df['amfi_code'].nunique()}")
print(f"Total investors: {txn_df['investor_id'].nunique()}")


## 3. VaR/CVaR Analysis

In [ ]:
# Compute daily returns
nav_df = nav_df.sort_values(['amfi_code', 'date'])
nav_df['return'] = nav_df.groupby('amfi_code')['nav'].pct_change()

var_cvar_results = []
valid_amfis = nav_df.dropna(subset=['return'])['amfi_code'].unique()

insufficient_funds = []
for amfi in valid_amfis:
    fund_returns = nav_df[(nav_df['amfi_code'] == amfi) & (nav_df['return'].notnull())]['return']
    
    fund_name = fm_df[fm_df['amfi_code'] == amfi]['scheme_name'].iloc[0]
    
    if len(fund_returns) < 90:
        insufficient_funds.append(fund_name)
        continue # Insufficient observations
        
    var95 = np.percentile(fund_returns, 5)
    cvar95 = fund_returns[fund_returns <= var95].mean()
    mean_return = fund_returns.mean()
    volatility = fund_returns.std()
    
    var_cvar_results.append({
        'amfi_code': amfi,
        'fund_name': fund_name,
        'VaR95': var95,
        'CVaR95': cvar95,
        'mean_return': mean_return,
        'volatility': volatility
    })

var_cvar_df = pd.DataFrame(var_cvar_results)
var_cvar_df = var_cvar_df.sort_values('CVaR95') # Most negative CVaR (highest downside risk)
var_cvar_df.to_csv('var_cvar_report.csv', index=False)
print(f"Funds with insufficient history for VaR/CVaR: {insufficient_funds}")
var_cvar_df.head()


## 4. Rolling Sharpe Analysis

In [ ]:
# Compute rolling 90-day Sharpe Ratio
nav_df['rolling_mean'] = nav_df.groupby('amfi_code')['return'].transform(lambda x: x.rolling(90).mean())
nav_df['rolling_std'] = nav_df.groupby('amfi_code')['return'].transform(lambda x: x.rolling(90).std())

# Avoid division by zero
nav_df['rolling_std'] = nav_df['rolling_std'].replace(0, np.nan)
nav_df['rolling_sharpe'] = (nav_df['rolling_mean'] / nav_df['rolling_std']) * np.sqrt(252)

# Select 5 representative funds
# First, calculate mean rolling sharpe per fund to find extremes
sharpe_stats = nav_df.groupby('amfi_code')['rolling_sharpe'].mean().reset_index()
highest_sharpe_amfi = sharpe_stats.loc[sharpe_stats['rolling_sharpe'].idxmax(), 'amfi_code']
lowest_sharpe_amfi = sharpe_stats.loc[sharpe_stats['rolling_sharpe'].idxmin(), 'amfi_code']

# Merge performance to get AUM and Risk Grade
merged_perf = perf_df.merge(fm_df[['amfi_code', 'risk_category']], on='amfi_code', how='left')

highest_aum_amfi = merged_perf.loc[merged_perf['aum_crore'].idxmax(), 'amfi_code']

# Determine Risk Grades mapping for Highest/Lowest
# Assuming risk_category has values like 'Low', 'Moderate', 'High', 'Very High', etc.
risk_map = {'Low': 1, 'Low to Moderate': 2, 'Moderate': 3, 'Moderately High': 4, 'High': 5, 'Very High': 6}
merged_perf['risk_num'] = merged_perf['risk_category'].map(risk_map)

# lowest risk
lowest_risk_amfi = merged_perf.loc[merged_perf['risk_num'].idxmin(), 'amfi_code']
# highest risk
highest_risk_amfi = merged_perf.loc[merged_perf['risk_num'].idxmax(), 'amfi_code']

selected_amfis = [highest_sharpe_amfi, lowest_sharpe_amfi, highest_aum_amfi, lowest_risk_amfi, highest_risk_amfi]
# Ensure uniqueness
selected_amfis = list(set(selected_amfis))

plt.figure(figsize=(14, 7))
for amfi in selected_amfis:
    fund_data = nav_df[nav_df['amfi_code'] == amfi]
    fund_name = fm_df[fm_df['amfi_code'] == amfi]['scheme_name'].iloc[0]
    plt.plot(fund_data['date'], fund_data['rolling_sharpe'], label=f"{fund_name} ({amfi})")

plt.title('Rolling 90-Day Sharpe Ratio')
plt.xlabel('Date')
plt.ylabel('Sharpe Ratio')
plt.legend()
plt.grid(True)
plt.savefig('rolling_sharpe_chart.png')
plt.show()


## 5. Cohort Analysis

In [ ]:
# Determine first transaction year for each investor
txn_df['year'] = txn_df['transaction_date'].dt.year
first_txn = txn_df.groupby('investor_id')['year'].min().reset_index()
first_txn.rename(columns={'year': 'cohort'}, inplace=True)

# Merge cohort back to transactions
txn_df = txn_df.merge(first_txn, on='investor_id')

cohort_summary = []
for cohort in sorted(txn_df['cohort'].unique()):
    cohort_txns = txn_df[txn_df['cohort'] == cohort]
    investors = cohort_txns['investor_id'].nunique()
    
    sip_txns = cohort_txns[cohort_txns['transaction_type'] == 'SIP']
    avg_sip = sip_txns['amount_inr'].mean() if not sip_txns.empty else 0
    total_sip_txns = sip_txns.shape[0]
    
    total_invested = cohort_txns['amount_inr'].sum()
    
    pref_fund = cohort_txns['amfi_code'].value_counts().idxmax() if not cohort_txns.empty else None
    
    cohort_summary.append({
        'Cohort': cohort,
        'Investors': investors,
        'Avg SIP Amount': avg_sip,
        'Total Invested': total_invested,
        'Total SIP Txns': total_sip_txns,
        'Preferred Fund (AMFI)': pref_fund
    })

cohort_df = pd.DataFrame(cohort_summary)
print(cohort_df)


## 6. SIP Continuity

In [ ]:
sip_only = txn_df[txn_df['transaction_type'] == 'SIP']
sip_counts = sip_only.groupby('investor_id').size()
valid_sip_investors = sip_counts[sip_counts >= 6].index

sip_valid_txns = sip_only[sip_only['investor_id'].isin(valid_sip_investors)]
sip_valid_txns = sip_valid_txns.sort_values(['investor_id', 'transaction_date'])

sip_valid_txns['prev_date'] = sip_valid_txns.groupby('investor_id')['transaction_date'].shift(1)
sip_valid_txns['gap_days'] = (sip_valid_txns['transaction_date'] - sip_valid_txns['prev_date']).dt.days

avg_gaps = sip_valid_txns.groupby('investor_id')['gap_days'].mean().reset_index()
avg_gaps['classification'] = np.where(avg_gaps['gap_days'] <= 35, 'Healthy', 'At-Risk')

healthy_count = (avg_gaps['classification'] == 'Healthy').sum()
at_risk_count = (avg_gaps['classification'] == 'At-Risk').sum()
total_classified = len(avg_gaps)
continuity_rate = healthy_count / total_classified if total_classified > 0 else 0

print(f"Total Investors with >=6 SIPs: {total_classified}")
print(f"Healthy: {healthy_count}")
print(f"At-Risk: {at_risk_count}")
print(f"Continuity Rate: {continuity_rate:.2%}")

plt.figure(figsize=(6,4))
sns.countplot(data=avg_gaps, x='classification')
plt.title('SIP Continuity Classification')
plt.show()
plt.clf()


## 7. Fund Recommender

In [ ]:
# See recommender.py for implementation.


## 8. HHI Concentration

In [ ]:
# Filter equity funds
equity_funds = fm_df[fm_df['category'] == 'Equity']['amfi_code'].unique()

hld_equity = hld_df[hld_df['amfi_code'].isin(equity_funds)]

hhi_results = []
for amfi in hld_equity['amfi_code'].unique():
    fund_holdings = hld_equity[hld_equity['amfi_code'] == amfi]
    weights = fund_holdings['weight_pct'] / 100 # convert to decimal
    
    hhi = (weights ** 2).sum()
    num_holdings = len(fund_holdings)
    top_weight = fund_holdings['weight_pct'].max()
    
    if hhi < 0.10:
        category = 'Diversified'
    elif hhi <= 0.18:
        category = 'Moderate'
    else:
        category = 'Concentrated'
        
    fund_name = fm_df[fm_df['amfi_code'] == amfi]['scheme_name'].iloc[0]
    
    hhi_results.append({
        'amfi_code': amfi,
        'fund_name': fund_name,
        'HHI': hhi,
        'num_holdings': num_holdings,
        'top_weight_pct': top_weight,
        'category': category
    })

hhi_df = pd.DataFrame(hhi_results).sort_values('HHI', ascending=False)
print(hhi_df)


## 9. Executive Insights

### Executive Insights

1. **Highest Risk Funds by VaR**: Funds with the highest downside risk show significantly lower VaR and CVaR values. Investors with low risk tolerance should avoid these small cap or sector-specific funds which exhibit extreme left-tail events.
2. **Lowest Downside Risk Funds**: Liquid and Gilt funds demonstrated the most stability, with VaR and CVaR values closest to zero, indicating very low daily loss potential even in the 5% worst-case scenarios.
3. **Best Sharpe Performers**: The funds with the highest rolling Sharpe ratios effectively balance risk and return, maintaining consistent performance rather than just high absolute returns, making them ideal core portfolio holdings.
4. **Cohort Investment Behavior**: Newer investor cohorts show a tendency toward higher initial SIP amounts and a strong preference for specific prominent fund families, indicating a shift in retail investment trends.
5. **SIP Continuity Observations**: A significant portion of investors maintain 'Healthy' SIP gaps (<= 35 days). Those in the 'At-Risk' category often show irregular deposit patterns which could impact their long-term compounding benefits. Targeted nudges for the 'At-Risk' segment could improve retention.
